# Data Analysis and Visualisation

## Lab 6: Describe the reviews

### Preliminaries

In [ ]:
import pandas as pd

### Part 1

#### Task 1: The mean and the median

In [ ]:
listings = pd.read_csv("data/listings_clean.csv", parse_dates=["last_scraped", "calendar_last_scraped", "first_review", "last_review"])

# Compare how mean and median are affected by outliers
small = pd.Series([3, 5, 8, 10, 12])
print("Mean and median of the original:", small.mean(), small.median())

with_one = pd.concat([small, pd.Series([1293])], ignore_index=True)
print("Mean and median with an outlier", with_one.mean(), with_one.median())

Mean and median of the original: 7.6 8.0
Mean and median with an outlier 221.83333333333334 9.0


One listing with 1293 reviews moves the mean from 7.6 to 221.8, but the median only from 8 to 9.

In [ ]:
# Calculating mean and median of the column "number_of_reviews"
reviews = listings["number_of_reviews"]
print(reviews.mean(), reviews.median())

35.75404459991255 9.0


In the column "number_of_reviews" the mean is 35.8 and the median 9. For "how many reviews does a typical listing have?" I'd use the median: half of the listings have 9 reviews or fewer. It is less sensitive to ouliers and so is a better indicator of a "typical value".

#### Task 2: Percentiles and the spread

In [ ]:
print(reviews.describe())

count    4574.000000
mean       35.754045
std        78.377200
min         0.000000
25%         2.000000
50%         9.000000
75%        29.000000
max      1293.000000
Name: number_of_reviews, dtype: float64


In [ ]:
# Calculate percentiles
print(reviews.quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))

0.10      0.00
0.25      2.00
0.50      9.00
0.75     29.00
0.90     94.00
0.99    400.27
Name: number_of_reviews, dtype: float64


In [ ]:
# Calculate IQR (interquartile range)
q1 = reviews.quantile(0.25)
q3 = reviews.quantile(0.75)
iqr = q3 - q1
print(q1, q3, iqr)

2.0 29.0 27.0


The IQR is 27 here: the middle half of the listings has between 2 and 29 reviews, so the spread of the typical listings is small compared with the maximum of 1293.

The 10th percentile is 0, so at least 10 % of the listings have no reviews at all.

#### Task 3: The fences and the outliers

In [ ]:
# Calculate the lower and upper fences
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr
print(f"Upper: {upper}, Lower: {lower}")

Upper: 69.5, Lower: -38.5


The lower fence is -38.5 and a number of reviews cannot be negative, so nothing can be below it.

In [ ]:
# How many listings are above the upper fence
above = listings[listings["number_of_reviews"] > upper]
print(len(above), round(len(above) / len(listings) * 100, 1))

582 12.7


In [ ]:
# How many listings of each room type above the upper fence
display(above["room_type"].value_counts())
# Compare median price of listings above upper fence to the listings below it
print("Median proce of listings above the fence (number of reviews):", above["price"].median())
print("Median proce of listings below the fence (number of reviews):", 
      listings.loc[listings["number_of_reviews"] <= upper, "price"].median())

room_type
Entire home/apt    446
Private room       121
Shared room         14
Hotel room           1
Name: count, dtype: int64

Median proce of listings above the fence (number of reviews): 1459.54
Median proce of listings below the fence (number of reviews): 1867.0


582 listings, or 12.7 %, are above the upper fence of 69 (there can't be a 0.5 reviews, so 69.5 must be iterpreted as "more than 69"). These listings are mostly  mostly "Entire home/apt" (446). Their median price of 1459.5 is lower than that of the other listings (1867.0), so the listings with the most reviews appear to be generally cheaper than the rest.

#### Task 4: Mark them and compare

In [ ]:
# Create new column marking ouliers among "number_of_reviews"
listings["reviews_outlier"] = listings["number_of_reviews"] > upper
print(listings["reviews_outlier"].sum())

582


In [ ]:
# Make statistical comparison with and without ouliers
without = listings.loc[~listings["reviews_outlier"], "number_of_reviews"]
comparison = pd.DataFrame({
    "all": reviews.agg(["count", "mean", "median", "std"]),
    "without": without.agg(["count", "mean", "median", "std"]),
})
# Relative change with outliers
comparison["change_%"] = ((comparison["without"] / comparison["all"] - 1) * 100).round(0)

print(comparison.round(1))

           all  without  change_%
count   4574.0   3992.0     -13.0
mean      35.8     12.9     -64.0
median     9.0      7.0     -22.0
std       78.4     15.5     -80.0


Without the 582 outliers the mean drops by 64 %, the standard deviation by 80 % and the median only by 22 %. The standard deviation changes the most and the median the least, which is why the median is the better number for typical listings.

#### Task 5: Describe the groups

In [ ]:
print(listings.groupby("room_type")["number_of_reviews"].describe().round(0))

                  count   mean    std   min   25%   50%    75%     max
room_type                                                             
Entire home/apt  3760.0   34.0   76.0   0.0   2.0   8.0   27.0  1293.0
Hotel room          3.0   61.0   16.0  49.0  52.0  56.0   68.0    79.0
Private room      787.0   41.0   84.0   0.0   2.0  12.0   38.0   867.0
Shared room        24.0  114.0  122.0   0.0  28.0  82.0  146.0   510.0


Shared rooms have the highest median (82), but with only 24 listings. Among the groups with enough listings, Private room has the highest median (12) vs Entire home/apt (8).

I would not say anything about Hotel room: it has only 3 listings.

#### Task 6: Correlation

In [ ]:
# Calculate correlations between "number_of_reviews" and other columns
columns = ["number_of_reviews", "price", "accommodates", "bedrooms", "availability_365",
           "review_scores_rating", "minimum_nights", "days_since_review"]
print(listings[columns].corr()["number_of_reviews"].drop("number_of_reviews").round(2))

price                  -0.09
accommodates           -0.05
bedrooms               -0.14
availability_365        0.18
review_scores_rating   -0.04
minimum_nights         -0.07
days_since_review      -0.19
Name: number_of_reviews, dtype: float64


All the correlations are weak (below 0.2 in size). The strongest is "days_since_review" at -0.19: listings with more reviews have a more recent last review. The weakest is "review_scores_rating" (-0.04). Everything is negative except availability_365 (0.18).

In [ ]:
# Calculate correlation with all listings
print(round(listings["number_of_reviews"].corr(listings["availability_365"]), 2))
# Calculate correlation without the outliers
print(round(without.corr(listings["availability_365"]), 2))

0.18
0.0


Without the outliers the correlation with availability_365 goes from 0.18 to 0.00, so that weak connection came entirely from the 582 listings with very many reviews. The first number says little about a typical listing.